# A/B Test Data Analysis: Revenue per User

An A/B test is a randomized experiment used to decide whether a new feature should be launched.
Real experiment data is rarely clean, and this notebook walks through several typical problems
(users in both groups, extreme outliers, many zero-revenue records, heavy-tailed revenue) and
how to handle them before testing for a difference between groups.

**Dataset:** the Kaggle "A/B Test Data" dataset (`AB_Test_Results.csv`), with columns
`USER_ID`, `VARIANT_NAME` (`control` / `variant`) and `REVENUE`.

### Contents
1. [Loading and cleaning the data](#1.-Loading-and-cleaning-the-data)
2. [Group metrics and distributions](#2.-Group-metrics-and-distributions)
3. [Statistical analysis](#3.-Statistical-analysis)
4. [Conclusions](#4.-Conclusions)

## 1. Loading and cleaning the data

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

# ---- configuration ----------------------------------------------------------
SEED = 42            # reproducible bootstrap
N_BOOT = 10_000      # number of bootstrap resamples
ALPHA = 0.05         # significance level
RNG = np.random.default_rng(SEED)

# The first path that exists is used (local repo, current folder, then Kaggle)
CANDIDATE_PATHS = [
    Path("data/AB_Test_Results.csv"),
    Path("AB_Test_Results.csv"),
    Path("/kaggle/input/ab-test-data/AB_Test_Results.csv"),
]

In [ ]:
data_path = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "AB_Test_Results.csv not found. Place it in ./data/ (see README) or edit CANDIDATE_PATHS."
    )

raw = pd.read_csv(data_path)
print(f"Loaded {data_path}  ->  {raw.shape[0]:,} rows, {raw.shape[1]} columns")
raw.head()

In [ ]:
raw.info()
raw.describe()

In [ ]:
print("Unique values per column:")
print(raw.nunique().to_string())
print(f"\nDuplicate rows: {raw.duplicated().sum():,}")
print(f"Missing values: {int(raw.isna().sum().sum())}")

### Users who appear in both groups

In a clean experiment every user belongs to exactly one group. Let's check how many users were assigned to more than one.

In [ ]:
groups_per_user = raw.groupby("USER_ID")["VARIANT_NAME"].nunique()
mixed_share = groups_per_user.value_counts().rename_axis("groups_per_user").to_frame("users")
mixed_share["share"] = (mixed_share["users"] / mixed_share["users"].sum()).round(4)
mixed_share

A large share of users saw both versions. Keeping them would blur the effect of the change
(their revenue cannot be attributed to a single group), so they are **excluded**.

In [ ]:
single_group_ids = groups_per_user[groups_per_user == 1].index
df = raw[raw["USER_ID"].isin(single_group_ids)].copy()

# sanity check: no user is left with two groups
assert df.groupby("USER_ID")["VARIANT_NAME"].nunique().max() == 1

removed_users = (groups_per_user > 1).sum()
print(f"Removed {removed_users:,} users in both groups "
      f"({removed_users / len(groups_per_user):.1%}); {len(df):,} rows remain.")

### Outliers

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x="VARIANT_NAME", y="REVENUE", data=df, ax=ax)
ax.set_title("Revenue per record, before outlier handling")
plt.show()

df.sort_values("REVENUE", ascending=False).head(10)

One record is far larger than the rest. Let's check whether that user has other extreme records,
or whether it is an isolated value.

In [ ]:
top_user = df.loc[df["REVENUE"].idxmax(), "USER_ID"]
print(f"User with the largest revenue record: {top_user}")
df[df["USER_ID"] == top_user]

It is a single isolated record. With no extra information to justify it as real behaviour, it is removed
because one value this large can dominate a mean-based comparison.

In [ ]:
df = df[df["USER_ID"] != top_user].copy()
print(f"Rows after outlier removal: {len(df):,}")

### From records to users

Most records have zero revenue, and a user can have both zero and positive records.
We treat records as *visits* and the **user as the experimental unit**: revenue is summed per user,
which gives one row per user.

In [ ]:
users_mixed = (
    df.groupby("USER_ID")["REVENUE"]
      .agg(has_zero=lambda s: (s == 0).any(), has_paid=lambda s: (s > 0).any())
)
print(f"Users with both zero and positive records: {(users_mixed.has_zero & users_mixed.has_paid).sum():,}")

users = (
    df.groupby(["USER_ID", "VARIANT_NAME"], as_index=False)
      .agg(REVENUE=("REVENUE", "sum"), VISITS=("REVENUE", "size"))
)
users["PAID"] = users["REVENUE"] > 0
print(f"Users in analysis: {len(users):,}")
users.head()

## 2. Group metrics and distributions

In [ ]:
def group_summary(frame):
    g = frame.groupby("VARIANT_NAME")
    out = pd.DataFrame({
        "users": g["USER_ID"].nunique(),
        "total_revenue": g["REVENUE"].sum(),
        "mean_revenue": g["REVENUE"].mean(),
        "median_revenue": g["REVENUE"].median(),
        "visits_per_user": g["VISITS"].mean(),
    })
    return out.round(4)

print("All users")
display(group_summary(users))

print("Paying users only")
display(group_summary(users[users["PAID"]]))

In [ ]:
conv = users.groupby("VARIANT_NAME")["PAID"].agg(users="size", payers="sum", conversion="mean")
conv["conversion"] = conv["conversion"].round(4)
conv

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.boxplot(x="VARIANT_NAME", y="REVENUE", data=users, ax=axes[0])
axes[0].set_title("Revenue per user (all users)")

sns.boxplot(x="VARIANT_NAME", y="REVENUE", data=users[users["PAID"]], ax=axes[1])
axes[1].set_title("Revenue per user (paying users)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

sns.histplot(data=users, x="REVENUE", hue="VARIANT_NAME", bins=50, stat="density",
             common_norm=False, element="step", ax=axes[0])
axes[0].set_title("Revenue distribution, all users")

sns.histplot(data=users[users["PAID"]], x="REVENUE", hue="VARIANT_NAME", bins=30, stat="density",
             common_norm=False, element="step", ax=axes[1])
axes[1].set_title("Revenue distribution, paying users")

plt.tight_layout()
plt.show()

## 3. Statistical analysis

### 3.1 Normality check

The plots show a very skewed, zero-heavy distribution. We confirm it formally with the Shapiro-Wilk test
for **both** groups (on a random subsample of at most 5,000 users, where the test is reliable).

In [ ]:
rows = []
for name, part in users.groupby("VARIANT_NAME"):
    sample = part["REVENUE"].sample(min(len(part), 5000), random_state=SEED)
    stat, p = stats.shapiro(sample)
    rows.append({"group": name, "n": len(sample), "W": round(stat, 4), "p_value": p})
pd.DataFrame(rows)

A p-value below 0.05 rejects normality in both groups, so a plain t-test is not the only tool we should rely on.

### 3.2 Mann-Whitney U test and Welch's t-test

The Mann-Whitney test compares distributions without assuming normality, but a sample with very many ties
(zeros) weakens its interpretation, so we use it together with Welch's t-test and the bootstrap below.

In [ ]:
def split(frame, paying_only=False):
    f = frame[frame["PAID"]] if paying_only else frame
    return (f.loc[f.VARIANT_NAME == "variant", "REVENUE"].to_numpy(),
            f.loc[f.VARIANT_NAME == "control", "REVENUE"].to_numpy())

results = []
for label, paying_only in [("all users", False), ("paying users", True)]:
    v, c = split(users, paying_only)
    mw = stats.mannwhitneyu(v, c, alternative="two-sided")
    tt = stats.ttest_ind(v, c, equal_var=False)
    results.append({
        "population": label,
        "n_variant": len(v), "n_control": len(c),
        "mean_diff (variant - control)": v.mean() - c.mean(),
        "mann_whitney_p": mw.pvalue,
        "welch_t_p": tt.pvalue,
    })

tests = pd.DataFrame(results).set_index("population")
tests.round(4)

### 3.3 Conversion rate (share of paying users): two-proportion z-test

In [ ]:
def two_prop_ztest(x1, n1, x2, n2):
    p1, p2 = x1 / n1, x2 / n2
    pooled = (x1 + x2) / (n1 + n2)
    se = np.sqrt(pooled * (1 - pooled) * (1 / n1 + 1 / n2))
    z = (p1 - p2) / se
    return p1 - p2, z, 2 * (1 - stats.norm.cdf(abs(z)))

cv = users[users.VARIANT_NAME == "variant"]["PAID"]
cc = users[users.VARIANT_NAME == "control"]["PAID"]
diff, z, p = two_prop_ztest(cv.sum(), len(cv), cc.sum(), len(cc))
print(f"Conversion  control: {cc.mean():.4%}   variant: {cv.mean():.4%}")
print(f"Difference (variant - control): {diff:+.4%}   z = {z:.3f}   p-value = {p:.4f}")

### 3.4 Bootstrap

To see the size and uncertainty of the difference in means, we resample each group with replacement
`N_BOOT` times. Resampling is done in chunks to keep memory use low.

In [ ]:
def bootstrap_means(values, n_boot=N_BOOT, chunk=500):
    values = np.asarray(values, dtype=float)
    out = np.empty(n_boot)
    for start in range(0, n_boot, chunk):
        end = min(start + chunk, n_boot)
        idx = RNG.integers(0, len(values), size=(end - start, len(values)))
        out[start:end] = values[idx].mean(axis=1)
    return out

def percentile_ci(samples, alpha=ALPHA):
    return np.percentile(samples, [100 * alpha / 2, 100 * (1 - alpha / 2)])

def bootstrap_p_value(diff_samples):
    # two-sided: how often the resampled difference falls on the other side of zero
    return min(1.0, 2 * min((diff_samples <= 0).mean(), (diff_samples >= 0).mean()))

boot = {}
for label, paying_only in [("all users", False), ("paying users", True)]:
    v, c = split(users, paying_only)
    boot[label] = {"variant": bootstrap_means(v), "control": bootstrap_means(c)}
print("Bootstrap done:", {k: len(v["variant"]) for k, v in boot.items()})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (label, d) in zip(axes, boot.items()):
    sns.kdeplot(d["control"], fill=True, label="control", ax=ax)
    sns.kdeplot(d["variant"], fill=True, label="variant", ax=ax)
    ax.set_title(f"Bootstrap distribution of the mean: {label}")
    ax.set_xlabel("mean revenue per user")
    ax.legend()
plt.tight_layout()
plt.show()

The distributions of the means overlap, but to judge the effect we need the distribution of the
**difference** between group means and its confidence interval.

In [ ]:
def plot_difference(diff_samples, title, alpha=ALPHA):
    lo, hi = percentile_ci(diff_samples, alpha)
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.kdeplot(diff_samples, fill=True, color="steelblue", ax=ax)
    ax.axvline(0, color="gray", lw=1.5, label="no difference (0)")
    ax.axvline(lo, color="crimson", ls="--")
    ax.axvline(hi, color="crimson", ls="--", label=f"{100 * (1 - alpha):.0f}% CI [{lo:.3f}, {hi:.3f}]")
    ax.set_title(f"Difference in mean revenue (variant - control): {title}")
    ax.set_xlabel("difference of means")
    ax.legend()
    plt.show()
    return lo, hi

summary_rows = []
for label, d in boot.items():
    diff_samples = d["variant"] - d["control"]
    lo, hi = plot_difference(diff_samples, label)
    summary_rows.append({
        "population": label,
        "mean_diff": diff_samples.mean(),
        "ci_low": lo, "ci_high": hi,
        "bootstrap_p": bootstrap_p_value(diff_samples),
        "zero_in_ci": lo <= 0 <= hi,
    })

boot_summary = pd.DataFrame(summary_rows).set_index("population")
boot_summary.round(4)

## 4. Conclusions

**Data preparation (applies to any run):**
- Users assigned to both groups were excluded so that the effect is attributed to one version only.
- The single most extreme revenue record was removed as an outlier.
- The user is the experimental unit: records were aggregated to one row per user.

**Result of this run** (generated from the numbers above):

In [ ]:
def verdict(p):
    return "statistically significant" if p < ALPHA else "NOT statistically significant"

print("Difference in mean revenue per user (variant - control)\n" + "-" * 56)
for label in boot_summary.index:
    r = boot_summary.loc[label]
    print(f"{label:>13}: {r.mean_diff:+.4f}  95% CI [{r.ci_low:+.4f}, {r.ci_high:+.4f}]  "
          f"bootstrap p={r.bootstrap_p:.3f}  -> {verdict(r.bootstrap_p)}")

print(f"\nConversion difference: {diff:+.4%} (p={p:.3f}) -> {verdict(p)}")
print(f"Mann-Whitney p-values: " + ", ".join(f"{k}: {v:.3f}" for k, v in tests["mann_whitney_p"].items()))
print(f"Welch t-test p-values: " + ", ".join(f"{k}: {v:.3f}" for k, v in tests["welch_t_p"].items()))

if (boot_summary["bootstrap_p"] >= ALPHA).all() and p >= ALPHA:
    print("\nNo reliable evidence that the variant changes revenue or conversion. "
          "Next steps: run an A/A test and a power analysis to see which effect size this setup can detect.")
else:
    print("\nAt least one metric shows a significant difference. Check it for practical relevance "
          "and consider correcting for multiple comparisons before shipping.")